# Validating Zwieback & Meyer 2022 Improvements 1 & 2 in dolphin

Reproduces, on simulated data, the comparison from Fig. 5 of
[Zwieback & Meyer 2022, *Reliable InSAR Phase History Uncertainty Estimates*](https://doi.org/10.1109/TGRS.2022.3146816).

Three CRLB-style estimators are compared against the actual Monte-Carlo RMSE
of the EMI phase estimate $\hat\theta$:

1. **Baseline expected partial FI** $\bar{\mathbf{K}}^p_{\theta\theta}$ &mdash; what dolphin currently returns from `compute_crlb_jax`.
2. **Improvement 1**: observed FI with Schur-complement marginalization over the magnitude block: $(\mathbf{F}_{\theta\theta} - \mathbf{F}_{\theta G}\mathbf{F}^+_{GG}\mathbf{F}^T_{\theta G})^+$ &mdash; new function `compute_observed_fi_crlb`.
3. **Improvement 1+2**: same as (2) but evaluated on the *penalized* sufficient statistic $\tilde C = C \circ W(\hat\gamma)$ (paper Eq. 12&ndash;13), which also shifts $\hat\theta$.

We sweep over `num_looks` $L \in \{100, 300, 500\}$ to cover the paper's L=100 fit and dolphin's typical operational range (L &approx; 200&ndash;500 with 7&ndash;15 px half-windows). The paper's published $W$ coefficients are fit at L=100 and are slightly over-conservative at higher L.

In [ ]:
from __future__ import annotations

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dolphin.phase_link import simulate as sim
from dolphin.phase_link.covariance import coh_mat_single
from dolphin.phase_link.crlb import (
    compute_crlb_jax,
    compute_observed_fi_crlb,
    penalization_weight,
)
from dolphin.phase_link.simulate import simulate_coh, simulate_neighborhood_stack

rng = np.random.default_rng(0)

## 1. Sanity-check the penalization weight $W(\hat\gamma)$

Compare against paper Fig. 3(b). The penalization should be near-total for
$\gamma < 0.05$, transition through $\gamma \approx 0.1$, and rejoin
$W \approx 1$ by $\gamma \approx 0.3$.

In [ ]:
gammas = np.linspace(0.001, 1.0, 400)
weights = []
for g in gammas:
    G = g * jnp.ones((2, 2)).at[jnp.arange(2), jnp.arange(2)].set(1.0)
    weights.append(float(penalization_weight(G)[0, 1]))
weights = np.asarray(weights)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(gammas, weights)
ax[0].set(xlabel=r'true $\gamma$', ylabel=r'$W(\gamma)$',
          title='Penalization weight (Z&M2022 Eq. 13)')
ax[0].axvspan(0, 0.05, color='r', alpha=0.1, label=r'$\gamma<0.05$: full penalization')
ax[0].axvspan(0.3, 1.0, color='g', alpha=0.1, label=r'$\gamma>0.3$: pass-through')
ax[0].legend(loc='center right', fontsize=8)
ax[0].grid(alpha=0.3)

ax[1].plot(gammas, gammas, 'k--', label='unpenalized', alpha=0.6)
ax[1].plot(gammas, gammas * weights, 'C0', label='penalized')
ax[1].set(xlabel=r'true $\gamma$', ylabel=r'effective $\gamma$ in $\tilde C$',
          title='Effect on coherence-magnitude entries')
ax[1].legend()
ax[1].grid(alpha=0.3)
fig.tight_layout()

## 2. Single-pixel EMI helper

Mirrors the EMI step inside `process_coherence_matrices`: smallest eigenvector of the Hadamard product $\mathbf\Gamma^{-1} \circ \mathbf C$, referenced to epoch 0.

In [ ]:
def run_emi(C: np.ndarray, ref_idx: int = 0, jitter: float = 1e-6) -> np.ndarray:
    """Single-pixel EMI; returns phase estimate of length N (radians)."""
    Gamma = np.abs(C)
    n = C.shape[0]
    Gamma_inv = np.linalg.solve(Gamma + jitter * np.eye(n), np.eye(n))
    M = Gamma_inv * C  # Hermitian Hadamard product
    _vals, vecs = np.linalg.eigh(M)
    v = vecs[:, 0]  # smallest eigenvalue first
    return np.angle(v) - np.angle(v[ref_idx])


def wrap(x: np.ndarray) -> np.ndarray:
    """Wrap phases to (-pi, pi]."""
    return np.angle(np.exp(1j * x))

## 3. Monte-Carlo harness

For a given true coherence matrix $\mathbf{C}_\text{true}$ and number of looks $L$, draws $R$ realizations of $L$ SLC samples, runs EMI on raw and penalized $C$, and computes:

- the predicted per-epoch standard deviation under each estimator (averaged across replicates), and
- the actual per-epoch RMSE of $\hat\theta$ relative to truth.

True phase is taken to be zero throughout (`add_signal=False`); the actual error is then just `wrap(theta_hat)`.

In [ ]:
import warnings


def run_mc(C_true: np.ndarray, L: int, R: int, seed: int = 0) -> dict:
    """Run R Monte-Carlo replicates and return predicted std + actual RMSE."""
    P = C_true.shape[0]
    sigma_base, sigma_imp1, sigma_imp12 = [np.zeros((R, P)) for _ in range(3)]
    err_base, err_imp12 = np.zeros((R, P)), np.zeros((R, P))

    sim.rng = np.random.default_rng(seed)
    # simulate_neighborhood_stack uses complex64 noise vs complex128 L; the
    # mixed-precision matmul triggers benign RuntimeWarnings ("divide by zero"
    # / "overflow") even when all outputs are finite. Silence them locally.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning,
                                message=r".*encountered in matmul.*")

        for r in range(R):
            samples = simulate_neighborhood_stack(C_true, neighbor_samples=L)
            C_emp = np.asarray(coh_mat_single(samples))

            # baseline + Imp 1 (operate on raw C)
            theta_base = run_emi(C_emp)
            sigma_base[r] = np.asarray(
                compute_crlb_jax(jnp.asarray(C_emp), num_looks=L, reference_idx=0,
                                 gamma_jitter=1e-6, fim_jitter=1e-6,
                                 mask_zero_blocks=False)
            )
            sigma_imp1[r] = np.asarray(
                compute_observed_fi_crlb(
                    jnp.asarray(C_emp)[None],
                    jnp.asarray(theta_base)[None],
                    num_looks=float(L),
                    reference_idx=0,
                )[0]
            )
            err_base[r] = wrap(theta_base)

            # Imp 1+2: penalized likelihood -> shifts both the EMI estimate and the FI
            W = np.asarray(penalization_weight(jnp.abs(jnp.asarray(C_emp))))
            C_pen = C_emp * W
            theta_pen = run_emi(C_pen)
            sigma_imp12[r] = np.asarray(
                compute_observed_fi_crlb(
                    jnp.asarray(C_pen)[None],
                    jnp.asarray(theta_pen)[None],
                    num_looks=float(L),
                    reference_idx=0,
                )[0]
            )
            err_imp12[r] = wrap(theta_pen)

    rmse_base = np.sqrt(np.mean(err_base**2, axis=0))
    rmse_imp12 = np.sqrt(np.mean(err_imp12**2, axis=0))
    return {
        "sigma_base": sigma_base.mean(axis=0),
        "sigma_imp1": sigma_imp1.mean(axis=0),
        "sigma_imp12": sigma_imp12.mean(axis=0),
        "rmse_base": rmse_base,
        "rmse_imp12": rmse_imp12,
    }


## 4. Reproduce Fig. 5 columns (moderate coherence)

Paper scenario: $P=16$, $\gamma_0 = 0.6$, $\gamma_\infty = 0.2$, exponential decay with $\tau_0 = 72$ days, $\Delta t = 12$ days. The paper used $L=100$, $R=1200$; we use the same $L$ first to verify we reproduce their result, then sweep.

In [ ]:
P = 16
C_mod, _ = simulate_coh(num_acq=P, gamma_inf=0.2, gamma0=0.6,
                       Tau0=72, acq_interval=12, add_signal=False)
C_mod = np.asarray(C_mod)

C_low, _ = simulate_coh(num_acq=P, gamma_inf=0.0, gamma0=0.6,
                       Tau0=72, acq_interval=12, add_signal=False)
C_low = np.asarray(C_low)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
im0 = ax[0].imshow(np.abs(C_mod), vmin=0, vmax=1, cmap='magma')
ax[0].set_title(r'moderate $\gamma$ ($\gamma_\infty=0.2$)')
plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(np.abs(C_low), vmin=0, vmax=1, cmap='magma')
ax[1].set_title(r'low $\gamma$ ($\gamma_\infty=0$)')
plt.colorbar(im1, ax=ax[1], fraction=0.046)
fig.tight_layout()

In [ ]:
R = 300  # paper used 1200; 300 is enough for clear trends in seconds
L_values = [100, 300, 500]

results = {}
for label, C_true in [('moderate', C_mod), ('low', C_low)]:
    results[label] = {}
    for L in L_values:
        print(f'  scenario={label:8s}  L={L:4d}  R={R}  ...', end=' ', flush=True)
        results[label][L] = run_mc(C_true, L=L, R=R, seed=L)
        print('done')

### 4.1 Per-epoch error vs. predicted std (Fig. 5 reproduction)

Each row is a coherence regime; columns step through $L$. Solid lines are the actual Monte-Carlo RMSE, dashed are the three CRLB estimators. The baseline (red dash) sits well below the actual line; Improvement 1 (orange) closes much of the gap; Improvement 1+2 (green) closes a bit more, especially at low $\gamma$.

In [ ]:
fig, axes = plt.subplots(2, len(L_values), figsize=(4 * len(L_values), 6.5),
                          sharex=True)
for row, label in enumerate(['moderate', 'low']):
    for col, L in enumerate(L_values):
        ax = axes[row, col]
        r = results[label][L]
        x = np.arange(P)
        ax.semilogy(x, np.degrees(r['rmse_base']),  'k-',  lw=2,
                    label='actual RMSE (raw EMI)')
        ax.semilogy(x, np.degrees(r['rmse_imp12']), 'k:',  lw=2,
                    label='actual RMSE (penalized EMI)')
        ax.semilogy(x, np.degrees(r['sigma_base']), 'C3--',
                    label=r'baseline $\bar K^p$')
        ax.semilogy(x, np.degrees(r['sigma_imp1']), 'C1--', label=r'Imp 1 $K^f$')
        ax.semilogy(x, np.degrees(r['sigma_imp12']),'C2--',
                    label=r'Imp 1+2 $K^f(\tilde C)$')
        ax.set_title(f'{label} $\\gamma$, L={L}')
        ax.grid(alpha=0.3)
        if col == 0:
            ax.set_ylabel(r'$\theta$ std [deg]')
        if row == 1:
            ax.set_xlabel('scene index')
        if row == 0 and col == 0:
            ax.legend(fontsize=8, loc='upper left')
fig.tight_layout()

### 4.2 Bias summary: median(predicted / actual) per (regime, L, estimator)

A ratio of 1.0 means the bound is tight; the paper's central finding is that the **baseline ratio is ~0.5 at moderate coherence and as low as ~0.1 at low coherence**, and that Improvements 1 and 2 push the ratio up toward 1. This table is the headline number that should drive the decision to flip dolphin's default.

In [ ]:
import pandas as pd

rows = []
for label in ['moderate', 'low']:
    for L in L_values:
        r = results[label][L]
        # exclude reference epoch where both predicted and actual are 0
        slc = slice(1, None)

        def ratio(pred, actual, slc=slc):
            """Median ratio of predicted to actual std over non-reference epochs."""
            return float(np.median(pred[slc] / np.maximum(actual[slc], 1e-9)))

        rows.append({
            'regime': label,
            'L': L,
            'baseline / actual': ratio(r['sigma_base'], r['rmse_base']),
            'imp1 / actual':     ratio(r['sigma_imp1'], r['rmse_base']),
            'imp1+2 / actual':   ratio(r['sigma_imp12'], r['rmse_imp12']),
        })

df = pd.DataFrame(rows).set_index(['regime', 'L'])
df.style.format('{:.2f}').background_gradient(cmap='RdYlGn', vmin=0.4, vmax=1.0)


### 4.3 Marginal benefit of Improvement 2 over Improvement 1

How much extra accuracy do we get by penalizing on top of the observed-FI Schur correction?  This is the key decision variable for whether to ship Improvement 2 alongside Improvement 1, or only Improvement 1.

In [ ]:
rows = []
for label in ['moderate', 'low']:
    for L in L_values:
        r = results[label][L]
        slc = slice(1, None)
        gap_imp1   = float(np.median(np.abs(
            r['sigma_imp1'][slc]   - r['rmse_base'][slc])))
        gap_imp12  = float(np.median(np.abs(
            r['sigma_imp12'][slc]  - r['rmse_imp12'][slc])))
        rows.append({
            'regime': label,
            'L': L,
            'gap |sigma - rmse| imp1 [rad]':  gap_imp1,
            'gap |sigma - rmse| imp1+2 [rad]': gap_imp12,
            'extra benefit from imp2 [rad]':  gap_imp1 - gap_imp12,
        })

df_gap = pd.DataFrame(rows).set_index(['regime', 'L'])
df_gap.style.format('{:.4f}')

## 5. Decision criteria

After running this notebook on representative scenarios, the PR should decide:

- **Always flip Improvement 1 to default-on** if `imp1 / actual` is closer to 1 than `baseline / actual` across all rows. The Z&M paper claims this should hold for any `L`; we are verifying it on dolphin's simulator and `compute_observed_fi_crlb` implementation.
- **Make Improvement 2 default-on** only if `extra benefit from imp2` is consistent and not negligible at the operational $L \in [200, 500]$. If the marginal gain at $L=300$&ndash;$500$ is below ~10% of typical RMSE, ship Improvement 2 as opt-in (`penalize_likelihood=True`) but leave the default off &mdash; modifying the EMI point estimate has follow-on implications for `temp_coh`, closure-phase, and compressed-SLC outputs that are not justified by a small bound improvement.

If the table in 4.2 shows that Improvement 1 alone reaches `>= 0.85` ratio at $L \geq 300$, then Improvement 2 is mostly cosmetic for our regime and should ship as opt-in only.